# STATS 21 Discussion 5 Solutions


In [1]:
import numpy as np
import pandas as pd


### 1. Matrix Standardization & Outlier Clipping
Write a function `standardize(M, axis=0, clip=None)` that takes in a 2D numpy array `M` and:
* if `axis=0`, standardizes each column to have mean 0 and standard deviation 1
* if `axis=1`, standardizes each row to have mean 0 and standard deviation 1
* if a column or row has standard deviation 0, sets that column or row to 0
* if `clip` is provided as a positive number `c`, clips all values to $[-c, c]$ using boolean masking

Do not use any loops (`for` or `while`).


In [2]:
def standardize(M, axis=0, clip=None):
    matrix = np.asarray(M, dtype=float)
    if axis == 0:
        mean = np.mean(matrix, axis=0)
        std = np.std(matrix, axis=0)
        std_safe = np.where(std == 0, 1.0, std)
        Z = (matrix - mean) / std_safe
        Z[:, std == 0] = 0.0
    elif axis == 1:
        mean = np.mean(matrix, axis=1, keepdims=True)
        std = np.std(matrix, axis=1, keepdims=True)
        std_safe = np.where(std == 0, 1.0, std)
        Z = (matrix - mean) / std_safe
        Z[std.flatten() == 0, :] = 0.0
    else:
        raise ValueError("axis must be 0 or 1")
    
    if clip is not None:
        Z[Z < -clip] = -clip
        Z[Z > clip] = clip
        
    return Z


In [3]:
M = np.array([
    [10.0, 20.0, 5.0],
    [20.0, 40.0, 5.0],
    [30.0, 60.0, 5.0]
])
standardize(M, axis=0)


In [4]:
standardize(M, axis=1)


In [5]:
standardize(M, axis=0, clip=1.0)


### 2. Gradebook Processing
The following DataFrame `df_grades` contains student scores and sections.

Write a function `process_grades(df)` that:
* creates a `'weighted'` column equal to `0.40 * midterm + 0.60 * final` using `.loc`
* adds 3 bonus points to students in Section `'A'` and 5 bonus points to students in Section `'B'` using `.loc`
* caps any scores exceeding 100 at 100 using `.loc`
* adds 4 bonus points to the student at index position 2 using `.iloc`
* extracts all students with `final >= 85` and `weighted >= 90`, returning only the columns `['section', 'weighted']` sorted by `'weighted'` descending


In [6]:
df_grades = pd.DataFrame({
    'midterm': [88, 92, 79, 95, 84, 62],
    'final': [91, 85, 88, 98, 76, 58],
    'section': ['A', 'B', 'A', 'B', 'A', 'B']
}, index=['Alice', 'Bob', 'Charlie', 'Diana', 'Evan', 'Frank'])

def process_grades(df):
    df = df.copy()
    df.loc[:, 'weighted'] = 0.40 * df.loc[:, 'midterm'] + 0.60 * df.loc[:, 'final']
    df.loc[df['section'] == 'A', 'weighted'] += 3.0
    df.loc[df['section'] == 'B', 'weighted'] += 5.0
    df.loc[df['weighted'] > 100.0, 'weighted'] = 100.0
    df.iloc[2, df.columns.get_loc('weighted')] += 4.0
    mask = (df['final'] >= 85) & (df['weighted'] >= 90.0)
    return df.loc[mask, ['section', 'weighted']].sort_values(by='weighted', ascending=False)


In [7]:
process_grades(df_grades)


,section,weighted
Diana,B,100.0
Alice,A,92.8
Bob,B,92.8
Charlie,A,91.4


### 3. Inventory Alignment & Data Cleaning
Write a function `reconcile_and_clean(s1, s2, catalog, sales)` that takes in two store inventory Series `s1` and `s2`, a `catalog` list, and a `sales` DataFrame:
* adds `s1` and `s2` using `.add()` with `fill_value=0` so missing items are treated as 0 instead of `NaN`
* reindexes the combined inventory to conform to `catalog`, setting missing products to 0 and converting values to `int`
* fills missing values in `sales['dept']` with the most frequent department using `.value_counts()`
* fills missing values in `sales['revenue']` with the column mean
* drops any rows where `commission` is missing using `.dropna()`
* returns a tuple `(catalog_inventory, cleaned_sales)`


In [8]:
store1 = pd.Series([10, 15, 8], index=['Python', 'R', 'SQL'])
store2 = pd.Series([5, 12, 20], index=['R', 'SQL', 'C++'])
catalog = ['Python', 'R', 'SQL', 'C++', 'Julia', 'Java']

sales_data = pd.DataFrame({
    'dept': ['Sales', 'HR', 'Sales', np.nan, 'IT', 'HR'],
    'revenue': [8500.0, np.nan, 9000.0, 9500.0, np.nan, 8000.0],
    'commission': [1000.0, 500.0, np.nan, 1200.0, 1100.0, 500.0]
})

def reconcile_and_clean(s1, s2, catalog, sales):
    total_inv = s1.add(s2, fill_value=0)
    catalog_inv = total_inv.reindex(catalog, fill_value=0).astype(int)
    
    cleaned_sales = sales.copy()
    mode_dept = cleaned_sales['dept'].value_counts().index[0]
    cleaned_sales['dept'] = cleaned_sales['dept'].fillna(mode_dept)
    cleaned_sales['revenue'] = cleaned_sales['revenue'].fillna(cleaned_sales['revenue'].mean())
    cleaned_sales = cleaned_sales.dropna(subset=['commission'])
    
    return catalog_inv, cleaned_sales


In [9]:
inv, cleaned = reconcile_and_clean(store1, store2, catalog, sales_data)
inv


AttributeError: 'Series' object has no attribute 'to_html'

In [10]:
cleaned


,dept,revenue,commission
0,Sales,8500.0,1000.0
1,HR,8750.0,500.0
3,Sales,9500.0,1200.0
4,IT,8750.0,1100.0
5,HR,8000.0,500.0
